# Семинар 2. Оптимизация и регуляризация

*Курс «Глубокое обучение», Центральный университет*

In [ ]:
%config Completer.use_jedi = False # Чтобы автодополнение с помощью табов работало
%load_ext autoreload
%autoreload 2

import torch
import numpy as np

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

## Оптимизация в PyTorch

На семинаре 1 мы научились считать в PyTorch градиенты, а из лекции 4 знаем: обучение сети — это задача оптимизации. В коде она обычно выглядит очень просто — несколько строк с `torch.optim` и один вызов `step()`. Но под капотом у оптимизатора есть состояние, группы параметров и собственные гиперпараметры.

Давай сначала научимся решать оптимизационную задачу на PyTorch как таковую, а во второй части семинара перейдём к обучению нейросетей.

### Задание: что делает оптимизатор и почему `zero_grad()` — часть алгоритма

В Семинаре 1 мы уже пользовались оптимизаторами (`SGD`, `Adam`) почти как «кнопкой обучения»: посчитали `loss`, вызвали `loss.backward()`, затем `optimizer.step()`. На этом месте легко поймать ощущение, что оптимизатор — это какая-то отдельная сущность, которая “сама” умеет улучшать параметры.

Чтобы снять это ощущение, полезно один раз проделать тот же процесс руками на простой задаче оптимизации **без нейросети**. Ниже используется функция Розенброка: это классическая “неудобная” поверхность, на которой методы оптимизации ведут себя по‑разному.

**Что сделать:**
1. Напиши «рукописный» градиентный спуск: на каждой итерации посчитай значение функции, сделай `backward()`, затем обнови параметры в блоке `torch.no_grad()`, и обязательно обнули градиенты (`x.grad = None` или `optimizer.zero_grad()`).
2. Повтори эксперимент, но уже с `torch.optim.SGD`: проверь, что смысл тот же, просто кода меньше.
3. Добавь `torch.optim.Adam` и сравни кривые сходимости. Важно не “кто красивее”, а **почему** на этой поверхности один метод может идти быстрее или стабильнее другого.

> Если у тебя медленная машина, можно временно уменьшить `num_iters`, но постарайся сохранить качественную картинку: важно увидеть тренд, а не одну случайную точку.


In [ ]:
# TODO (задание):
# 1) Реализуй «рукописный» GD: forward -> backward -> шаг под torch.no_grad() -> сброс grad
# 2) Повтори то же самое через torch.optim.SGD
# 3) Добавь torch.optim.Adam и сравни динамику
#
# Советы:
#  - обновление параметров делайте под torch.no_grad(), чтобы не раздувать граф вычислений
#  - не забывайте обнулять градиенты, иначе они будут накапливаться и оптимизация пойдёт "не туда"



Ниже — аккуратный вариант, который сначала делает ручной GD, а затем повторяет тот же эксперимент через `torch.optim`.


In [ ]:
def rosenbrock(x, a=1, b=100):
    return torch.sum((a - x[:-1]) ** 2 + b * (x[1:] - x[:-1] ** 2) ** 2)

> Ремарка: функция Розенброка имеет глобальный минимум в точке `(1, 1, ..., 1)`.  
> Это удобно: ты можешь не только смотреть на кривую loss, но и проверять, насколько близко параметры подошли к вектору единиц.


In [ ]:
num_iters = 10000
dim = 5

x = torch.full((dim, ), 1/ dim, requires_grad=True)
lr = 3e-4

values = []

for i in range(num_iters):
    value = rosenbrock(x)
    value.backward()

    with torch.no_grad():
        x.data = x.data - lr * x.grad
    x.grad = None
    values.append(value.item())

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_formats = ['svg']


plt.plot(torch.arange(num_iters), values)
plt.grid(True)
plt.xlabel('Номер итерации')
plt.ylabel('Значение функции Розенброка')
plt.show()

In [ ]:
x

А что, если нам понадобится написать сложный оптимизатор, хотя бы Adam? Придётся всё делать руками? А если параметров много? То плодить циклы? К счастью, нет. В PyTorch есть отдельный модуль, который отвечает за оптимизаторы. Посмотрим, как работать с ними.

In [ ]:
from torch import optim

In [ ]:
x = torch.full((dim, ), 1/ dim, requires_grad=True)
optimizer = optim.SGD([x], lr=lr)

values_pt = []

for i in range(num_iters):
    value = rosenbrock(x)
    value.backward()
    optimizer.step()
    optimizer.zero_grad()
    values_pt.append(value.item())

In [ ]:
plt.plot(torch.arange(num_iters), values, label='Рукописный GD')
plt.plot(torch.arange(num_iters), values_pt, label='PyTorch GD')
plt.grid(True)
plt.xlabel('Номер итерации')
plt.ylabel('Значение функции Розенброка')
plt.legend()
plt.show()

In [ ]:
x = torch.full((dim, ), 1/ dim, requires_grad=True)
optimizer = optim.Adam([x], lr=lr)

values_adam = []

for i in range(num_iters):
    value = rosenbrock(x)
    value.backward()
    optimizer.step()
    optimizer.zero_grad()
    values_adam.append(value.item())

In [ ]:
plt.plot(torch.arange(num_iters), values, label='Рукописный GD')
plt.plot(torch.arange(num_iters), values_pt, label='PyTorch GD')
plt.plot(torch.arange(num_iters), values_adam, label='Adam')
plt.grid(True)
plt.xlabel('Номер итерации')
plt.ylabel('Значение функции Розенброка')
plt.legend()
plt.show()

#### Итог после задания

Здесь важно унести две мысли, которые напрямую переходят в обучение нейросетей.

1. **Оптимизатор не делает ничего “сверхъестественного”.** В основе всё равно лежит градиент (то, что попало в `.grad` после `backward()`), а шаг — это просто правило обновления параметров. Разница между методами (`SGD`, `Adam` и т.д.) — в том, *как* именно они используют историю градиентов и масштабы координат, чтобы выбрать размер и направление шага.

2. **Обнуление градиентов — часть алгоритма.** PyTorch по умолчанию аккумулирует градиенты в `.grad`. Это удобно, когда ты сознательно хочешь накопить градиент от нескольких мини‑батчей, но в обычном обучении приводит к тому, что “градиент за шаг” превращается в сумму градиентов за много шагов и оптимизация начинает вести себя странно.



Однако `torch.optim` содержит далеко не все оптимизаторы. И это может быть неприятно. Как тогда написать свой?

In [ ]:
class Signum(optim.Optimizer):
    def __init__(self, params, lr=0.01, momentum=0.09, weight_decay=0, **kwargs):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay)

        super(Signum, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(Signum, self).__setstate__(state)

    def step(self, closure=None):

        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            weight_decay = group['weight_decay']
            momentum = group['momentum']

            for p in group['params']:
                if p.grad is None:
                    continue
                d_p = p.grad.data
                if weight_decay != 0:
                    d_p.add_(p.data, alpha=weight_decay)
                if momentum != 0:
                    # signum
                    param_state = self.state[p]
                    if 'momentum_buffer' not in param_state:
                        buf = param_state['momentum_buffer'] = torch.zeros_like(p.data)

                    else:
                        buf = param_state['momentum_buffer']

                    buf.mul_(momentum).add_(d_p, alpha=(1 - momentum))
                    d_p = torch.sign(buf)
                else:#signsgd
                    d_p = torch.sign(d_p)

                p.data.add_(d_p, alpha=-group['lr'])

        return loss

In [ ]:
x = torch.full((dim, ), 1/ dim, requires_grad=True)
optimizer = Signum([x], lr=lr)

values_signum = []

for i in range(num_iters):
    value = rosenbrock(x)
    value.backward()
    optimizer.step()
    optimizer.zero_grad()
    values_signum.append(value.item())

In [ ]:
plt.plot(torch.arange(num_iters), values, label='Рукописный GD')
plt.plot(torch.arange(num_iters), values_pt, label='PyTorch GD')
plt.plot(torch.arange(num_iters), values_adam, label='Adam')
plt.plot(torch.arange(num_iters), values_signum, label='Signum')
plt.grid(True)
plt.xlabel('Номер итерации')
plt.ylabel('Значение функции Розенброка')
plt.legend()
plt.show()

Можно заметить, что мы везде для получения `lr` и `momentum` использовали group. Зачем такое может быть нужно? Это даёт возможность разбить параметры модели на группы и использовать разные параметры оптимизатора для каждого из тензоров. Иногда это может помочь лучше обучить модель. Разберём игрушечный пример.

In [ ]:
x = torch.full((dim, ), 1/ dim, requires_grad=True)
z = torch.full((dim, ), 0.2/ dim, requires_grad=True)
optimizer = optim.SGD([x, z], lr=lr)

values = []

for i in range(num_iters):
    value = rosenbrock(x * z ** 2)
    value.backward()
    optimizer.step()
    optimizer.zero_grad()
    values.append(value.item())

print(f"Минимальное найденное значение псевдо функции Розенброка: {min(values)}")

In [ ]:
x = torch.full((dim, ), 1/ dim, requires_grad=True)
z = torch.full((dim, ), 0.2/ dim, requires_grad=True)
optimizer = optim.SGD([
    {'params': x},
    {'params': z, 'lr': 1e-3}
], lr=lr)

values = []

for i in range(num_iters):
    value = rosenbrock(x * z ** 2)
    value.backward()
    optimizer.step()
    optimizer.zero_grad()
    values.append(value.item())

print(f"Минимальное найденное значение псевдо функции Розенброка: {min(values)}")

---
# Оптимизация, регуляризация и инициализация на практике


## Обозначения

Коротко о буквах, которые встретятся ниже:

- $f_{w}(\mathbf{x})$ — нейросеть с параметрами $w$; $\ell(y, \mathbf{z})$ — loss на объекте;
- шаг оптимизации: $w_{t+1}=w_t-\eta_t g_t$, где $g_t$ — градиент по мини‑батчу $B_t$;
- $\lambda$ — коэффициент weight decay; $\delta$ — маленькая константа для численной устойчивости (в PyTorch — `eps`).

В коде ниже:
- `lr` соответствует $\eta$,
- `weight_decay` соответствует $\lambda$,
- `dropout_p` — это $p_{drop}$ (вероятность «выкинуть» активацию).

## Задача и функция потерь в этом семинаре (MNIST)

- Число классов: $K=10$.
- Модель возвращает **логиты** $\mathbf{z}=f_w(\mathbf{x})\in\mathbb{R}^K$ (в коде это переменная `logits`).
- Метка $y$ — **индекс** класса (в PyTorch: $y\in\{0,\ldots,K-1\}$), **не one‑hot**.
- `F.cross_entropy(logits, y)` реализует softmax + NLL:
  - $\mathbf{p}=\mathrm{softmax}(\mathbf{z})$, $\;p_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}$,
  - $\ell(y,\mathbf{z})=-\log p_y$.
- Предсказанный класс: $\hat{y}=\arg\max_k z_k$ (в коде `logits.argmax(dim=1)`).

> Эти обозначения совпадают со словарём из лекций.


Мы будем смотреть на две кривые:
- **train loss/accuracy** — как модель подстраивается под обучающие данные,
- **val loss/accuracy** — как она переносит знания на новые данные.

<div style="padding:12px; border-left: 4px solid #D32F2F; background: #FFEBEE;">
<b>Быстрая диагностика по графикам</b><br/>
1) <b>Недообучение (underfitting)</b>: train и val плохие и почти не улучшаются → чаще всего не хватает модели/эпох/правильного lr.<br/>
2) <b>Переобучение (overfitting)</b>: train улучшается, а val портится → добавляем регуляризацию / данных / уменьшаем модель / early stopping.<br/>
3) <b>Слишком большой lr</b>: loss прыгает, растёт, появляются NaN → уменьшить lr (обычно в 3–10 раз).<br/>
4) <b>Слишком маленький lr</b>: всё улучшается, но очень медленно → увеличить lr или добавить scheduler.<br/>
</div>

Дальше мы намеренно создадим ситуацию, где переобучение видно быстро (будем обучаться на уменьшенной части MNIST).


## Оптимизация

В лекции мы уже видели, что оптимизатор задаёт **траекторию** в пространстве параметров $w$.

На практике чаще всего важно не «какая формула красивее», а:
- адекватный **learning rate** и его **расписание**,
- правильный **оптимизатор под задачу**,
- стабильность градиентов (иногда нужен gradient clipping).

<div style="padding:12px; border-left: 4px solid #2E7D32; background: #E8F5E9;">
<b>Правила большого пальца (очень грубо)</b><br/>
- Начать можно с <b>AdamW</b>: lr ≈ $10^{-3}$, weight decay $λ$ ≈ $10^{-4}$ (дальше тюнить).<br/>
- Для <b>SGD+momentum</b>: lr часто значительно больше (например, $0.1$), momentum $γ$ ≈ $0.9$.<br/>
- Если батч растёт, иногда можно увеличить lr (но это не «железное» правило).<br/>
</div>

Ниже мы используем один и тот же «стенд» (MLP на MNIST) и будем менять только настройки оптимизации/регуляризации.


In [ ]:
%matplotlib inline

import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from typing import Optional

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:


from torch.utils.data import Subset

def get_mnist_loaders(
    batch_size: int = 256,
    augment: bool = False,
    train_size: Optional[int] = 5000,
    val_size: int = 10000,
    split_seed: int = 42,
    num_workers: int = 2,
):
    """
    MNIST loaders with an optional small train subset (so that overfitting appears quickly).

    train_size:
      - None  -> use all available train samples (except validation split)
      - int   -> use only that many samples from the train split
    """
    # Normalization constants for MNIST
    mean, std = (0.1307,), (0.3081,)

    train_tf = [transforms.ToTensor(), transforms.Normalize(mean, std)]
    if augment:
        # Light, label-preserving transforms for MNIST
        train_tf = [
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=5,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]

    train_tf = transforms.Compose(train_tf)
    eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

    # Two dataset objects to allow different transforms for train and val
    full_train = datasets.MNIST(root="./data", train=True, download=True, transform=train_tf)
    full_eval = datasets.MNIST(root="./data", train=True, download=True, transform=eval_tf)

    n_total = len(full_train)
    assert 0 < val_size < n_total, "val_size must be between 1 and len(MNIST)-1"
    n_train = n_total - val_size

    # Fixed split for reproducibility
    g = torch.Generator().manual_seed(split_seed)
    perm = torch.randperm(n_total, generator=g).tolist()
    train_idx = perm[:n_train]
    val_idx = perm[n_train:]

    if train_size is not None:
        train_idx = train_idx[:train_size]
    train_ds = Subset(full_train, train_idx)
    val_ds = Subset(full_eval, val_idx)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, len(train_ds), len(val_ds)


In [ ]:
import torch.nn.init as init

class TwoLayerMLP(nn.Module):
    """
    2-слойный MLP:
    x (784) -> Linear -> (BatchNorm1d) -> ReLU -> (Dropout) -> Linear -> logits(10)

    Важно:
    - BatchNorm и Dropout ведут себя по-разному в train() и eval() режимах.
    - Инициализация тоже важна: «слишком маленькие» или «слишком большие» веса могут сделать обучение нестабильным.
    """
    def __init__(self, hidden_dim: int = 256, dropout_p: float = 0.0, use_bn: bool = True):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden_dim)

        self.use_bn = use_bn
        self.bn1 = nn.BatchNorm1d(hidden_dim) if use_bn else nn.Identity()

        self.dropout = nn.Dropout(p=dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc2 = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)          # (B, 784)
        x = self.fc1(x)                    # (B, H)
        x = self.bn1(x)                    # (B, H)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)                    # (B, 10)
        return x


def apply_initialization(model: nn.Module, scheme: str = "default", nonlinearity: str = "relu") -> None:
    """
    Явная инициализация весов.

    scheme:
      - "default"  : ничего не делаем (оставляем дефолт PyTorch)
      - "xavier"   : Xavier/Glorot (хорошо для tanh/sigmoid и иногда для линейных частей)
      - "kaiming"  : He/Kaiming (обычно лучший старт для ReLU)
      - "tiny"     : намеренно слишком маленькие веса (для демонстрации затухания)
      - "large"    : намеренно слишком большие веса (для демонстрации взрыва)

    nonlinearity: используется в Kaiming инициализации (например, "relu").
    """
    scheme = (scheme or "default").lower()

    if scheme in ("default", "none"):
        return

    for m in model.modules():
        if isinstance(m, nn.Linear):
            if scheme in ("xavier", "glorot"):
                init.xavier_uniform_(m.weight)
            elif scheme in ("kaiming", "he"):
                init.kaiming_normal_(m.weight, nonlinearity=nonlinearity)
            elif scheme == "tiny":
                init.normal_(m.weight, mean=0.0, std=1e-3)
            elif scheme == "large":
                init.normal_(m.weight, mean=0.0, std=10.0)
            else:
                raise ValueError(f"Unknown init scheme: {scheme}")

            if m.bias is not None:
                init.zeros_(m.bias)

        if isinstance(m, nn.BatchNorm1d):
            # стандартно: γ_BN=1, β_BN=0
            if m.weight is not None:
                init.ones_(m.weight)
            if m.bias is not None:
                init.zeros_(m.bias)


# Быстрая проверка, что модель создаётся
model = TwoLayerMLP(hidden_dim=256, dropout_p=0.0, use_bn=False).to(device)
sum(p.numel() for p in model.parameters())


In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")

        total_loss += loss.item()
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = total_correct / total
    return avg_loss, acc


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    clip_grad_norm: Optional[float] = None,
):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()


        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        optimizer.step()

        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / total
    acc = total_correct / total
    return avg_loss, acc


In [ ]:
from IPython.display import clear_output
import time

def render_figures(history, cfg, suffix: str = ""):
    clear_output(wait=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    ax0 = axes[0]
    ax0.plot(history["train_loss"], label="train")
    ax0.plot(history["val_loss"], label="val")
    ax0.set_title(f"{cfg['name']} | Loss{suffix}")
    ax0.set_xlabel("epoch")
    ax0.set_ylabel("loss")
    ax0.grid(True)
    ax0.legend()

    if cfg.get("loss_xlim") is not None:
        ax0.set_xlim(cfg["loss_xlim"])
    if cfg.get("loss_ylim") is not None:
        ax0.set_ylim(cfg["loss_ylim"])

    # Accuracy
    ax1 = axes[1]
    ax1.plot(history["train_acc"], label="train")
    ax1.plot(history["val_acc"], label="val")
    ax1.set_title(f"{cfg['name']} | Accuracy{suffix}")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("accuracy")
    ax1.grid(True)
    ax1.legend()

    if cfg.get("acc_xlim") is not None:
        ax1.set_xlim(cfg["acc_xlim"])
    if cfg.get("acc_ylim") is not None:
        ax1.set_ylim(cfg["acc_ylim"])

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def build_optimizer(model: nn.Module, cfg: dict):
    name = cfg.get("optimizer", "adamw").lower()
    lr = float(cfg["lr"])
    weight_decay = float(cfg.get("weight_decay", 0.0))

    if name == "sgd":
        momentum = float(cfg.get("momentum", 0.9))
        nesterov = bool(cfg.get("nesterov", False))
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            nesterov=nesterov,
            weight_decay=weight_decay,  # for SGD this is equivalent to L2 penalty
        )

    if name == "adam":
        # In torch.optim.Adam, weight_decay is implemented as L2 penalty (coupled with adaptive scaling)
        return torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )

    if name == "adamw":
        # AdamW = decoupled weight decay (often preferable)
        return torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )

    raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(optimizer: torch.optim.Optimizer, cfg: dict):
    sched = cfg.get("scheduler", None)
    if sched is None:
        return None

    sched = str(sched).lower()
    if sched == "steplr":
        step_size = int(cfg.get("step_size", 5))
        gamma = float(cfg.get("lr_gamma", 0.1))
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    if sched == "cosine":
        # Simple cosine schedule over epochs
        T_max = int(cfg.get("T_max", cfg["epochs"]))
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max)

    raise ValueError(f"Unknown scheduler: {sched}")


def run_experiment(cfg: dict):
    """
    cfg keys (main):
      - name: str
      - epochs: int
      - batch_size: int
      - hidden_dim: int
      - lr: float
      - optimizer: {"adamw","adam","sgd"}
      - weight_decay: float
      - dropout_p: float
      - augment: bool
      - use_bn: bool
      - init: str | None   # "default" | "xavier" | "kaiming" | "tiny" | "large"
      - train_size: int | None
      - clip_grad_norm: float | None
      - scheduler: None | {"steplr","cosine"}
    """
    train_loader, val_loader, n_train, n_val = get_mnist_loaders(
        batch_size=cfg["batch_size"],
        augment=cfg["augment"],
        train_size=cfg.get("train_size", 5000),
        val_size=cfg.get("val_size", 10000),
        split_seed=cfg.get("split_seed", 42),
    )
    print(f"Dataset sizes: train={n_train}, val={n_val}")

    if "deep" in cfg["name"]:
      model = DeepMLP(
          hidden_dim=cfg["hidden_dim"],
      ).to(device)
    else:
      model = TwoLayerMLP(
          hidden_dim=cfg["hidden_dim"],
          dropout_p=cfg["dropout_p"],
          use_bn=cfg.get("use_bn", True),
      ).to(device)

    # Optional: override default PyTorch init
    init_scheme = cfg.get("init", "default")
    apply_initialization(model, init_scheme, nonlinearity="relu")
    print(f"Init scheme: {init_scheme}")

    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    live_plot = bool(cfg.get("live_plot", True))
    plot_every = int(cfg.get("plot_every", 1))
    clip_grad_norm = cfg.get("clip_grad_norm", None)

    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, clip_grad_norm=clip_grad_norm)
        va_loss, va_acc = evaluate(model, val_loader)

        # current lr (assume one param group)
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(current_lr)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        print(
            f"[{cfg['name']}] epoch {epoch:02d}/{cfg['epochs']} | "
            f"lr {current_lr:.2e} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

        if scheduler is not None:
            scheduler.step()

        if live_plot and (epoch % plot_every == 0 or epoch == 1):
            render_figures(history, cfg, "")

    return history


### Задание: базовый запуск (проверяем, что всё работает)

Сделаем «голую» модель без регуляризации (dropout/BN/augmentation выключены) и посмотрим:
- переобучается ли она на маленьком train‑подмножестве,
- как выглядят кривые train/val.

> Обрати внимание: мы используем `train_size=5000`, чтобы переобучение проявилось быстро.

Запусти ячейку ниже, затем попробуй:
- увеличить `epochs`,
- поменять `lr` (например, `3e-4`, `1e-3`, `3e-3`),
- включить `scheduler="cosine"` или `scheduler="steplr"` (опционально).


In [ ]:

cfg = dict(
    name="baseline_adamw",
    epochs=20,
    batch_size=256,
    train_size=256,   # чтобы быстрее увидеть overfitting
    val_size=10000,
    hidden_dim=1024,

    optimizer="adamw",
    lr=1e-2,
    weight_decay=0.0,

    dropout_p=0.0,
    use_bn=False,
    augment=False,

    # stability tricks
    clip_grad_norm=None,

    # optional schedulers: None | "cosine" | "steplr"
    scheduler=None,
    # for StepLR
    step_size=5,
    lr_gamma=0.2,

    live_plot=True,
    plot_every=1,
    loss_ylim=(0.0, 4),
    acc_ylim=(0.3, 1.0),
)

set_seed(42)
history_baseline = run_experiment(cfg);


## Инициализация и стабильность обучения

В лекции мы добавили раздел про **инициализацию**: почему при плохих начальных весах
активации и градиенты могут **затухать** или **взрываться**.

Цель здесь — увидеть это на практике:
1) сравнить, как ведёт себя *глубокая* MLP при разных схемах инициализации;
2) посмотреть на норму градиента на самом первом шаге;
3) (опционально) проверить, как это отражается на кривых обучения.

Подсказка:
- для ReLU обычно подходит **He/Kaiming**;
- Xavier чаще используют для tanh/sigmoid.


In [ ]:
class DeepMLP(nn.Module):
    """Глубокая MLP для демонстрации затухания/взрыва по глубине."""
    def __init__(self, input_dim: int = 28 * 28, hidden_dim: int = 256, depth: int = 8, num_classes: int = 10):
        super().__init__()
        dims = [input_dim] + [hidden_dim] * depth + [num_classes]
        self.layers = nn.ModuleList([nn.Linear(dims[i], dims[i + 1]) for i in range(len(dims) - 1)])

    def forward(self, x, return_acts: bool = False):
        x = x.view(x.size(0), -1)
        acts = []
        for lin in self.layers[:-1]:
            x = lin(x)
            x = F.relu(x)
            acts.append(x)
        x = self.layers[-1](x)
        if return_acts:
            return x, acts
        return x


@torch.no_grad()
def collect_activation_stds(model: nn.Module, x: torch.Tensor):
    """Собираем mean/std активаций (после ReLU) по слоям."""
    model.eval()
    _, acts = model(x, return_acts=True)
    means = [a.float().mean().item() for a in acts]
    stds = [a.float().std().item() for a in acts]
    return means, stds


def grad_norm_one_batch(model: nn.Module, x: torch.Tensor, y: torch.Tensor):
    """Норма градиента на одном батче (после backward)."""
    model.train()
    for p in model.parameters():
        p.grad = None

    logits = model(x)
    loss = F.cross_entropy(logits, y)
    loss.backward()

    total_sq = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_sq += float(p.grad.detach().pow(2).sum().item())

    return math.sqrt(total_sq), float(loss.item())


In [ ]:
# Берём один батч реальных данных (MNIST) и смотрим на статистики "на старте"
set_seed(42)
probe_train_loader, _, _, _ = get_mnist_loaders(batch_size=256, augment=False, train_size=2048, val_size=10000)
x0, y0 = next(iter(probe_train_loader))
x0, y0 = x0.to(device), y0.to(device)

schemes = ["tiny", "kaiming", "large"]
depth = 10

plt.figure(figsize=(8, 4))
for scheme in schemes:
    set_seed(42)
    m = DeepMLP(hidden_dim=256, depth=depth).to(device)
    apply_initialization(m, scheme)
    means, stds = collect_activation_stds(m, x0)
    plt.plot(stds, label=scheme)

plt.title(f"DeepMLP: std активаций после ReLU по слоям (depth={depth})")
plt.xlabel("индекс слоя")
plt.ylabel("std")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()

print("Градиенты на первом шаге (до какого-либо обучения):")
for scheme in schemes:
    set_seed(42)
    m = DeepMLP(hidden_dim=256, depth=depth).to(device)
    apply_initialization(m, scheme)
    gn, loss0 = grad_norm_one_batch(m, x0, y0)
    print(f"init={scheme:>7} | loss@step0={loss0:.3f} | grad_norm@step0={gn:.2e}")


## Регуляризация и устойчивость обобщения

Даже если оптимизация «идёт», модель может **переобучаться**: train становится хорошим, а val — нет.

Регуляризация — это не только «добавить L2/weight decay в loss». В DL это целый набор приёмов, которые:
- уменьшают переобучение,
- делают обучение устойчивее,
- иногда ускоряют сходимость (BatchNorm).

Ниже один и тот же стенд, но разные «ручки».


### 3.1 Weight decay (λ) и AdamW

В лекции (Мини‑пример 8) обсуждалось, что для адаптивных оптимизаторов важно различать:
- **L2‑штраф** (когда добавка вида `λ * w` фактически «попадает» внутрь градиентного шага),
- **decoupled weight decay** (AdamW), где «усадка» параметров отделена от градиента по данным.

В коде ниже:
- `torch.optim.Adam(..., weight_decay=...)` реализует **L2‑штраф**,
- `torch.optim.AdamW(..., weight_decay=...)` — **decoupled weight decay**.

<div style="padding:12px; border-left: 4px solid #1976D2; background: #E3F2FD;">
<b>Задание:</b> включи небольшой weight decay и сравни с baseline.<br/>
Ожидаемо: train может стать чуть хуже, но val обычно улучшается (меньше overfitting).<br/>
</div>


In [ ]:
cfg = dict(
    name="adamw_weight_decay",
    epochs=12,
    batch_size=256,
    train_size=5000,
    val_size=10000,
    hidden_dim=256,

    optimizer="adamw",
    lr=1e-3,
    weight_decay=1e-4,   # <-- lambda

    dropout_p=0.0,
    use_bn=False,
    augment=False,

    clip_grad_norm=None,
    scheduler=None,

    live_plot=True,
    plot_every=1,
    loss_ylim=(0.0, 0.50),
    acc_ylim=(0.85, 1.0),
)

set_seed(42)
history_wd = run_experiment(cfg);


### 3.2 Dropout

Идея: на этапе обучения случайно «выключаем» часть нейронов (активаций), чтобы сеть **не могла опираться только на одну "понравившуюся" ей комбинацию признаков**.  
Это снижает *co-adaptation* (взаимную “подгонку” нейронов друг под друга) и обычно уменьшает переобучение.

#### Что происходит математически (inverted dropout)
Пусть на каком-то слое получились активации (вектор) $\mathbf{h}$.

1) Сэмплируем бинарную маску $\mathbf{m}$ поэлементно:
- $\mathbf{m}_j$ принимает значение 0 или 1,
- $\mathbb{P}(\mathbf{m}_j = 1) = p_{keep}$, где $p_{keep}=1-p_{drop}$.

2) Применяем маску и **масштабируем** (это и есть “inverted dropout”):
- $\mathbf{h}_{drop} = (\mathbf{m} ⊙ \mathbf{h}) / p_{keep}$

Тогда в среднем масштаб не “плывёт”:
- $\mathbb{E}[\mathbf{h}_{drop}] = \mathbf{h}$

> В PyTorch `nn.Dropout(p=...)` как раз делает *inverted dropout*: на `train()` зануляет и делит на $p_{keep}$, а на `eval()` — просто возвращает вход.

#### Почему это работает
Dropout можно представить как обучение **множества “подсетей”**, которые разделяют веса.  
На `eval()` мы используем “усреднённую” модель (все нейроны включены), поэтому часто получаем лучшую обобщающую способность.

#### Где ставить dropout
Практические схемы (для MLP):
- без BatchNorm: `Linear → ReLU → Dropout → Linear → ...`
- с BatchNorm: `Linear → BatchNorm → ReLU → Dropout → ...`  
  (важно: dropout **после** BN, иначе статистики BN будут сильно шуметь)

Для CNN чаще используют `Dropout2d`/`SpatialDropout` (зануляют целые каналы) и ставят ближе к “голове” (классификатору), а не после каждого свёрточного слоя.

#### Как выбирать $p_{drop}$
- начни с небольшого: `0.1–0.2` (как в этом семинаре),
- если переобучение сильное — повышай (`0.3–0.5` для полносвязных частей),
- если модель **недообучается** (и train плохой) — уменьшай dropout или отключай.

#### Типичные ошибки
- забыли `model.eval()` перед валидацией/инференсом → dropout остаётся включённым, метрики “пляшут”;
- слишком большой dropout → обучение замедляется и может уйти в underfitting;
- ставить dropout *до* BatchNorm → BN получает лишний шум в статистиках.

<div style="padding:12px; border-left: 4px solid #2E7D32; background: #E8F5E9;">
<b>Практика:</b> dropout часто помогает, когда модель явно переобучается (train растёт, val падает).<br/>
Ожидаемо: train станет чуть хуже, а val — лучше (или стабильнее). Если стало хуже и там, и там — вероятно, dropout слишком сильный.<br/>
</div>


In [ ]:
cfg = dict(
    name="dropout",
    epochs=12,
    batch_size=256,
    train_size=5000,
    val_size=10000,
    hidden_dim=256,

    optimizer="adamw",
    lr=1e-3,
    weight_decay=0.0,

    dropout_p=0.1,   # <- p_drop = 1 - p_keep
    use_bn=False,
    augment=False,

    clip_grad_norm=None,
    scheduler=None,

    live_plot=True,
    plot_every=1,
    loss_ylim=(0.0, 0.50),
    acc_ylim=(0.85, 1.0),
)

set_seed(42)
history_dropout = run_experiment(cfg);


### 3.3 BatchNorm

BatchNorm (BN) нормирует активации **по мини‑батчу**, а затем применяет обучаемый масштаб/сдвиг.  
Это часто делает обучение **стабильнее** и менее чувствительным к инициализации и learning rate.

#### Формула (для одной компоненты/канала)
Пусть на вход BN пришли значения $\{x_i\}_{i\in B_t}$ (одна и та же “координата” активации для объектов текущего батча $B_t$, где $|B_t|=m$).

1) Считаем статистики по батчу:
- $\mu_{B_t} = \frac{1}{|B_t|} \sum\limits_{i\in B_t} x_i$
- $\sigma_{B_t}^2 = \frac{1}{|B_t|} \sum\limits_{i\in B_t} (x_i - \mu_{B_t})^2$

2) Нормируем ($\delta$ — маленькая константа для стабильности; **в PyTorch она называется `eps`**):
- $\hat{x}_i = \frac{x_i - \mu_{B_t}}{\sqrt{\sigma_{B_t}^2 + \delta}}$

3) Возвращаем “гибкость” через обучаемое аффинное преобразование:
- $y_i = \gamma_{BN}\,\hat{x}_i + \beta_{BN}$

Параметры $\gamma_{BN}$ и $\beta_{BN}$ обучаются вместе с остальными весами (то есть входят в общий вектор параметров $w$).

> В векторном виде BN делает это **по каждой размерности признака отдельно**.  
> В `BatchNorm2d` (CNN) статистики считаются по каждому каналу, усредняя по измерениям батча и пространству (N, H, W).

#### Почему BN помогает оптимизации
- выравнивает масштаб активаций → градиенты обычно более “ровные”;
- позволяет чаще использовать более высокий learning rate;
- вносит небольшой шум (из‑за статистик по батчу) → может работать как регуляризатор.

#### Train vs Eval: самый важный практический момент
BN ведёт себя по‑разному в `model.train()` и `model.eval()`:

- на `train()` статистики берутся из текущего батча ($\mu_{B_t}, \sigma_{B_t}^2$) и **обновляются running‑оценки**  
  (в PyTorch скорость обновления задаётся параметром `momentum`; здесь обозначим её $\rho$):
  - $\mu_{run} \leftarrow (1-\rho)\,\mu_{run} + \rho\,\mu_{B_t}$
  - $\sigma_{run}^2 \leftarrow (1-\rho)\,\sigma_{run}^2 + \rho\,\sigma_{B_t}^2$

- на `eval()` нормировка делается по “накопленным” ($\mu_{run}, \sigma_{run}^2$), чтобы предсказание **не зависело от состава батча**.

> Поэтому если забыть `model.eval()` на валидации, метрики могут стать хуже и/или нестабильными.

#### Когда BN может не помогать
- очень маленький batch size → оценки $\mu_{B_t}, \sigma_{B_t}^2$ становятся шумными, эффект может ухудшиться;
- в таких случаях часто пробуют: увеличить batch, уменьшить lr, или перейти на нормализации без зависимости от батча (LayerNorm/GroupNorm — обсудим позже).

<div style="padding:12px; border-left: 4px solid #1976D2; background: #E3F2FD;">
<b>Задание:</b> включи BatchNorm и сравни кривые и финальный val.<br/>
Дополнительно: попробуй уменьшить `batch_size` — иногда при маленьком батче BN начинает работать хуже.<br/>
</div>


In [ ]:
cfg = dict(
    name="batchnorm",
    epochs=12,
    batch_size=256,
    train_size=5000,
    val_size=10000,
    hidden_dim=256,

    optimizer="adamw",
    lr=1e-3,
    weight_decay=0.0,

    dropout_p=0.0,
    use_bn=True,
    augment=False,

    clip_grad_norm=None,
    scheduler=None,

    live_plot=True,
    plot_every=1,
    loss_ylim=None,
    acc_ylim=(0.85, 1.0),
)

set_seed(42)
history_bn = run_experiment(cfg);


### 3.4 Аугментации данных

Аугментации (data augmentation) — способ увеличить эффективный объём данных, применяя преобразования, которые **не меняют класс**.

Для MNIST подойдут лёгкие аффинные преобразования: повороты, сдвиги, масштабирование.

<div style="padding:12px; border-left: 4px solid #2E7D32; background: #E8F5E9;">
<b>Задание:</b> включи аугментации и сравни с baseline.<br/>
Часто: train станет хуже (задача сложнее), но val станет лучше.<br/>
</div>


In [ ]:
cfg = dict(
    name="augmentation",
    epochs=12,
    batch_size=256,
    train_size=5000,
    val_size=10000,
    hidden_dim=256,

    optimizer="adamw",
    lr=1e-3,
    weight_decay=0.0,

    dropout_p=0.0,
    use_bn=False,
    augment=True,

    clip_grad_norm=None,
    scheduler=None,

    live_plot=True,
    plot_every=1,
    loss_ylim=(0.0, 0.80),
    acc_ylim=(0.80, 1.0),
)

set_seed(42)
history_aug = run_experiment(cfg);


### 3.5 Комбинация техник

На практике часто работают комбинации:
- AdamW + weight decay,
- (опционально) BatchNorm,
- (опционально) Dropout,
- аугментации.

> Это не «универсальный рецепт». Иногда BatchNorm и Dropout вместе дают хуже, чем по отдельности — поэтому важно проверять кривые.

Запусти конфиг и сравни с предыдущими.


In [ ]:
cfg = dict(
    name="wd+dropout+bn+aug",
    epochs=12,
    batch_size=256,
    train_size=5000,
    val_size=10000,
    hidden_dim=256,

    optimizer="adamw",
    lr=1e-3,
    weight_decay=1e-4,

    dropout_p=0.1,
    use_bn=True,
    augment=True,

    clip_grad_norm=None,
    scheduler=None,

    live_plot=True,
    plot_every=1,
    loss_ylim=(0.0, 0.80),
    acc_ylim=(0.80, 1.0),
)

set_seed(42)
history_combo = run_experiment(cfg);


## (Опционально) Конфиги экспериментов

Когда экспериментов с гиперпараметрами становится много, их удобно выносить из кода в конфиг. Два стандартных инструмента --- `ml_collections.ConfigDict` и `Hydra`. Для примера с Hydra рядом с ноутбуком должен лежать файл `hydra_example.yaml`.

### Конфиги


Существуют аккуратные способы управления конфигами, например использование `ml_collection.Config_Dict` или же библиотеки `Hydra`. Ниже разберем пример на основе `ml_collection` конфига из известной статьи.

In [ ]:
!pip install ml_collections -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.5 MB/s eta 0:00:00


In [ ]:
import ml_collections

In [ ]:
# пример взят https://github.com/yang-song/score_sde_pytorch/blob/cb1f359f4aadf0ff9a5e122fe8fffc9451fd6e44/configs/default_cifar10_configs.py#L5

def get_default_configs():
    config = ml_collections.ConfigDict()
    # training
    config.training = training = ml_collections.ConfigDict()
    config.training.batch_size = 128
    training.n_iters = 1300001
    training.snapshot_freq = 50000
    training.log_freq = 50
    training.eval_freq = 100
    ## store additional checkpoints for preemption in cloud computing environments
    training.snapshot_freq_for_preemption = 10000
    ## produce samples at each snapshot.
    training.snapshot_sampling = True
    training.likelihood_weighting = False
    training.continuous = True
    training.reduce_mean = False

    # evaluation
    config.eval = evaluate = ml_collections.ConfigDict()
    evaluate.begin_ckpt = 9
    evaluate.end_ckpt = 26
    evaluate.batch_size = 1024
    evaluate.enable_sampling = False
    evaluate.num_samples = 50000
    evaluate.enable_loss = True
    evaluate.enable_bpd = False
    evaluate.bpd_dataset = 'test'

    # data
    config.data = data = ml_collections.ConfigDict()
    data.dataset = 'CIFAR10'
    data.image_size = 32
    data.random_flip = True
    data.centered = False
    data.uniform_dequantization = False
    data.num_channels = 3

    # model
    config.model = model = ml_collections.ConfigDict()
    model.sigma_min = 0.01
    model.sigma_max = 50
    model.num_scales = 1000
    model.beta_min = 0.1
    model.beta_max = 20.
    model.dropout = 0.1
    model.embedding_type = 'fourier'

    # optimization
    config.optim = optim = ml_collections.ConfigDict()
    optim.weight_decay = 0
    optim.optimizer = 'Adam'
    optim.lr = 2e-4
    optim.beta1 = 0.9
    optim.eps = 1e-8
    optim.warmup = 5000
    optim.grad_clip = 1.

    config.seed = 42
    config.device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

    return config

get_default_configs()

data:
  centered: false
  dataset: CIFAR10
  image_size: 32
  num_channels: 3
  random_flip: true
  uniform_dequantization: false
device: !!python/object/apply:torch.device
- cuda
- 0
eval:
  batch_size: 1024
  begin_ckpt: 9
  bpd_dataset: test
  enable_bpd: false
  enable_loss: true
  enable_sampling: false
  end_ckpt: 26
  num_samples: 50000
model:
  beta_max: 20.0
  beta_min: 0.1
  dropout: 0.1
  embedding_type: fourier
  num_scales: 1000
  sigma_max: 50
  sigma_min: 0.01
optim:
  beta1: 0.9
  eps: 1.0e-08
  grad_clip: 1.0
  lr: 0.0002
  optimizer: Adam
  warmup: 5000
  weight_decay: 0
seed: 42
training:
  batch_size: 128
  continuous: true
  eval_freq: 100
  likelihood_weighting: false
  log_freq: 50
  n_iters: 1300001
  reduce_mean: false
  snapshot_freq: 50000
  snapshot_freq_for_preemption: 10000
  snapshot_sampling: true

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

In [ ]:
# Пример взят по ссылке
# https://github.com/CompVis/latent-diffusion/blob/a506df5756472e2ebaf9078affdde2c4f1502cd4/configs/autoencoder/autoencoder_kl_32x32x4.yaml#L1

!cat ./hydra_example.yaml

model:
  base_learning_rate: 4.5e-6
  target: ldm.models.autoencoder.AutoencoderKL
  params:
    monitor: "val/rec_loss"
    embed_dim: 4
    lossconfig:
      target: ldm.modules.losses.LPIPSWithDiscriminator
      params:
        disc_start: 50001
        kl_weight: 0.000001
        disc_weight: 0.5

    ddconfig:
      double_z: True
      z_channels: 4
      resolution: 256
      in_channels: 3
      out_ch: 3
      ch: 128
      ch_mult: [ 1,2,4,4 ]  # num_down = len(ch_mult)-1
      num_res_blocks: 2
      attn_resolutions: [ ]
      dropout: 0.0

data:
  target: main.DataModuleFromConfig
  params:
    batch_size: 12
    wrap: True
    train:
      target: ldm.data.imagenet.ImageNetSRTrain
      params:
        size: 256
        degradation: pil_nearest
    validation:
      target: ldm.data.imagenet.ImageNetSRValidation
      params:
        size: 256
        degradation: pil_nearest

lightning:
  callbacks:
    image_logger:
      target: main.ImageLogger
      params:
        

In [ ]:
with initialize(config_path='./', version_base=None):
    config = compose(config_name="hydra_example.yaml")

config

{'model': {'base_learning_rate': 4.5e-06, 'target': 'ldm.models.autoencoder.AutoencoderKL', 'params': {'monitor': 'val/rec_loss', 'embed_dim': 4, 'lossconfig': {'target': 'ldm.modules.losses.LPIPSWithDiscriminator', 'params': {'disc_start': 50001, 'kl_weight': 1e-06, 'disc_weight': 0.5}}, 'ddconfig': {'double_z': True, 'z_channels': 4, 'resolution': 256, 'in_channels': 3, 'out_ch': 3, 'ch': 128, 'ch_mult': [1, 2, 4, 4], 'num_res_blocks': 2, 'attn_resolutions': [], 'dropout': 0.0}}}, 'data': {'target': 'main.DataModuleFromConfig', 'params': {'batch_size': 12, 'wrap': True, 'train': {'target': 'ldm.data.imagenet.ImageNetSRTrain', 'params': {'size': 256, 'degradation': 'pil_nearest'}}, 'validation': {'target': 'ldm.data.imagenet.ImageNetSRValidation', 'params': {'size': 256, 'degradation': 'pil_nearest'}}}}, 'lightning': {'callbacks': {'image_logger': {'target': 'main.ImageLogger', 'params': {'batch_frequency': 1000, 'max_images': 8, 'increase_log_steps': True}}}, 'trainer': {'benchmark':